# Сравнение MM-алгоритмов по сценариям

| Компонент | Значение |
|---|---|
| Алгоритмы | 6 (от baseline до RL) |
| Сценарии | 7 (от stationary до структурных шоков) |
| Seeds | 25 |
| Итого симуляций | 1050 |

In [26]:
%matplotlib inline
import os, sys, json, shutil, subprocess
from pathlib import Path

PROJECT_ROOT = Path(os.path.abspath(os.path.join(os.getcwd(), '..')))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np

from src.analysis import load

LOGS_BASE = PROJECT_ROOT / 'experiments' / 'log'
EXPERIMENTS_DIR = PROJECT_ROOT / 'experiments'
SCENARIO_SCRIPT = PROJECT_ROOT / 'src' / 'scenarios' / 'multimarket_baseline.py'
VENDOR_ABIDES = PROJECT_ROOT / 'vendor' / 'abides'

RUN_PREFIX = 'cmp'
FORCE_RERUN = False     # True = пересчитать всё
MAX_PARALLEL = 3

## Конфигурация

In [27]:
COMMON_ARGS = [
    '--num-noise', '500',
    '--num-noise-fx', '3000',
    '--num-value', '25',
]

SEEDS = list(range(42, 67))   # 25 x симуляций

# Путь к обученному DQN-чекпойнту
RL_POLICY_PATH = PROJECT_ROOT / 'experiments' / 'rl' / 'latest_best.pt'

#   baseline           — фикс-spread, без inventory mgmt
#   spread_based_tight — ladder фикс-spread, без inventory mgmt
#   adaptive           — ladder + sigmoid size-skew
#   as                 — Cross-Market Avellaneda-Stoikov (Bergault 2021)
#   glft               — Cross-Market GLFT (stationary, без T-t scaling)
#   rl_glft            — DQN-policy выбирает γ для GLFT динамически

MMS = {
    'baseline': {
        'mm_type': 'baseline',
        'extra_args': [],
        'a_usd_agent_name': 'BaselineMM_A_USD',
        'b_eur_agent_name': 'BaselineMM_B_EUR',
    },
    'spread_based_tight': {
        'mm_type': 'spread_based',
        'extra_args': [
            '--sb-window-size', '2',
            '--sb-num-ticks', '10',
            '--sb-order-size', '5',
        ],
        'a_usd_agent_name': 'SpreadBasedMM_A_USD',
        'b_eur_agent_name': 'SpreadBasedMM_B_EUR',
    },
    'adaptive': {
        'mm_type': 'adaptive',
        'extra_args': [
            '--adp-window-size', '2',
            '--adp-num-ticks', '10',
            '--adp-skew-beta', '1.0',
            '--adp-min-order-size', '5',
        ],
        'a_usd_agent_name': 'AdaptiveMM_A_USD',
        'b_eur_agent_name': 'AdaptiveMM_B_EUR',
    },
    'as': {
        'mm_type': 'as',
        'extra_args': ['--as-gamma', '5e-8'],
        'a_usd_agent_name': 'AvellanedaStoikovMM_A_USD',
        'b_eur_agent_name': 'AvellanedaStoikovMM_B_EUR',
    },
    'glft': {
        'mm_type': 'glft',
        'extra_args': ['--glft-gamma', '5e-7', '--glft-A', '0.3'],
        'a_usd_agent_name': 'GLFTMM_A_USD',
        'b_eur_agent_name': 'GLFTMM_B_EUR',
    },
}

if RL_POLICY_PATH.exists():
    MMS['rl_glft'] = {
        'mm_type': 'rl_glft',
        'extra_args': ['--rl-policy-path', str(RL_POLICY_PATH), '--rl-A', '0.3'],
        'a_usd_agent_name': 'RLGLFTMM_A_USD',
        'b_eur_agent_name': 'RLGLFTMM_B_EUR',
    }
else:
    pass

# 7 сценариев — покрытие разных типов шоков:
#   stationary        — без шоков (контроль)
#   megashock         — частые jumps fundamental'а (волатильный режим)
#   drift_down / _up  — устойчивый тренд вниз/вверх
#   news_shock        — одноразовый jump fundamental'а
#   liquidity_crisis  — снос bid-side стакана A_USD
#   fx_shock          — резкий buy FX (cross-market шок через FX-mid)
SCENARIOS = {
    'stationary':       [],
    'megashock':        ['--stress-multiplier', '1e6', '--megashock-mean', '100'],
    'drift_down':       ['--drift-shock', '10:00-10:15:-500'],
    'drift_up':         ['--drift-shock', '10:00-10:15:+500'],
    'news_shock':       ['--news-shock', '10:00:+300'],
    'liquidity_crisis': ['--liquidity-shock', 'A_USD:10:00:bid'],
    'fx_shock':         ['--fx-shock', '10:00:2000:buy'],
}

n_total = len(MMS) * len(SCENARIOS) * len(SEEDS)
print(f'Total runs: {n_total}')

Total runs: 1050


## Запуск симуляций

In [28]:
def _validate_run(log_dir, mm_key):
    """Лог A_USD MM содержит STARTING_CASH и ENDING_CASH_USD_EQUIV → True."""
    mm_cfg = MMS[mm_key]
    try:
        log = load.load_agent_log(log_dir, mm_cfg['a_usd_agent_name'], base=LOGS_BASE)
        if (log['EventType'] == 'STARTING_CASH').sum() == 0:
            return False
        has_end = ((log['EventType'] == 'ENDING_CASH_USD_EQUIV').sum()
                   + (log['EventType'] == 'ENDING_CASH').sum()) > 0
        return has_end
    except Exception:
        return False


def _build_cli_args(mm_key, scenario_key, seed):
    """CLI-args для одного (mm, scenario, seed) — сигнатура для cache-check."""
    mm_cfg = MMS[mm_key]
    return [
        '--seed', str(seed),
        '--mm-type', mm_cfg['mm_type'],
        *mm_cfg['extra_args'],
        *SCENARIOS[scenario_key],
        *COMMON_ARGS,
        '--fast-mode',
    ]


def _meta_matches(log_path, cli_args):
    meta_file = log_path / '_meta.json'
    if not meta_file.exists():
        return False
    try:
        return json.loads(meta_file.read_text()) == cli_args
    except Exception:
        return False


def run_scenario(mm_key, scenario_key, seed, force=False):
    """Запускает одну (mm, scenario, seed). Возвращает (log_dir, status)."""
    log_dir = f'{RUN_PREFIX}_{mm_key}_{scenario_key}_s{seed}'
    log_path = LOGS_BASE / log_dir
    cli_args = _build_cli_args(mm_key, scenario_key, seed)

    if log_path.exists() and not force:
        if _meta_matches(log_path, cli_args) and _validate_run(log_dir, mm_key):
            return log_dir, 'cached'
        shutil.rmtree(log_path)

    cmd = ['python', str(SCENARIO_SCRIPT), *cli_args, '--log-dir', log_dir]
    env = os.environ.copy()
    env['PYTHONPATH'] = f'{VENDOR_ABIDES}:{PROJECT_ROOT}'

    result = subprocess.run(cmd, cwd=str(EXPERIMENTS_DIR), env=env,
                            capture_output=True, text=True, timeout=600)
    if result.returncode != 0:
        print(f'FAILED (rc!=0): {mm_key} × {scenario_key} × s{seed}')
        print(result.stderr[-1000:])
        return log_dir, 'failed'

    if not _validate_run(log_dir, mm_key):
        if log_path.exists():
            shutil.rmtree(log_path)
        return log_dir, 'broken'

    (log_path / '_meta.json').write_text(json.dumps(cli_args))
    return log_dir, 'ok'

In [29]:
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

results = {}
jobs = [(mm, sc, seed) for mm in MMS for sc in SCENARIOS for seed in SEEDS]
total = len(jobs)
t0 = time.time()


print(f'=== Pass 1: parallel (MAX_PARALLEL={MAX_PARALLEL}), {total} jobs ===')
with ThreadPoolExecutor(max_workers=MAX_PARALLEL) as ex:
    futures = {ex.submit(run_scenario, mm, sc, seed, FORCE_RERUN): (mm, sc, seed)
               for mm, sc, seed in jobs}
    done = 0
    for fut in as_completed(futures):
        key = futures[fut]
        log_dir, status = fut.result()
        results[key] = (log_dir, status)
        done += 1
        if done % 25 == 0 or done == total or status not in ('ok', 'cached'):
            elapsed = time.time() - t0
            mm, sc, seed = key
            print(f'[{done:4d}/{total}] {mm:20s} × {sc:18s} × s{seed} {status:7s} '
                  f'(elapsed {elapsed:.0f}s)')


need_retry = [k for k, (_, s) in results.items() if s in ('broken', 'failed')]
if need_retry:
    print(f'\n=== Pass 2: sequential retry ({len(need_retry)} runs) ===')
    for i, key in enumerate(need_retry, 1):
        mm, sc, seed = key
        log_dir, status = run_scenario(mm, sc, seed, force=True)
        results[key] = (log_dir, status)
        print(f'[{i}/{len(need_retry)}] {mm:20s} × {sc:18s} × s{seed} {status:7s}')


still_failed = [k for k, (_, s) in results.items() if s in ('broken', 'failed')]
ok_count = sum(1 for _, (_, s) in results.items() if s in ('ok', 'cached'))
print(f'\n{ok_count}/{total} runs ok. Total time: {time.time() - t0:.0f}s')
if still_failed:
    print(f'!!! {len(still_failed)} STILL FAILED: {still_failed}')

=== Pass 1: parallel (MAX_PARALLEL=3), 1050 jobs ===
[  25/1050] baseline             × megashock          × s42 cached  (elapsed 0s)
[  50/1050] baseline             × megashock          × s66 cached  (elapsed 0s)
[  75/1050] baseline             × drift_down         × s66 cached  (elapsed 0s)
[ 100/1050] baseline             × news_shock         × s42 cached  (elapsed 1s)
[ 125/1050] baseline             × liquidity_crisis   × s42 cached  (elapsed 1s)
[ 150/1050] baseline             × liquidity_crisis   × s65 cached  (elapsed 1s)
[ 175/1050] baseline             × fx_shock           × s66 cached  (elapsed 1s)
[ 200/1050] spread_based_tight   × megashock          × s42 cached  (elapsed 1s)
[ 225/1050] spread_based_tight   × megashock          × s66 cached  (elapsed 1s)
[ 250/1050] spread_based_tight   × drift_down         × s66 cached  (elapsed 1s)
[ 275/1050] spread_based_tight   × news_shock         × s42 cached  (elapsed 2s)
[ 300/1050] spread_based_tight   × news_shock         × 

## Подготовка результатов

In [30]:
def get_pnl(run_name, agent_name):
    """Возвращает {'starting','ending','pnl','pnl_pct'} или None при ошибке."""
    try:
        log = load.load_agent_log(run_name, agent_name, base=LOGS_BASE)
    except Exception:
        return None
    s = log.loc[log['EventType'] == 'STARTING_CASH', 'Event']
    e = log.loc[log['EventType'] == 'ENDING_CASH_USD_EQUIV', 'Event']
    if s.empty or e.empty:
        return None
    sv, ev = float(s.iloc[0]), float(e.iloc[0])
    return {'starting': sv, 'ending': ev, 'pnl': ev - sv, 'pnl_pct': (ev - sv) / sv * 100}


def total_mm_volume(run, agent):
    """Σ|qty| из TRADE events — для расчёта post-hoc rebate."""
    try:
        log = load.load_agent_log(run, agent, base=LOGS_BASE)
    except Exception:
        return 0
    trades = log[log['EventType'] == 'TRADE']
    if trades.empty:
        return 0
    return sum(abs(int(ev['qty'])) for ev in trades['Event'] if isinstance(ev, dict))

In [31]:
POST_HOC_REBATE_CENTS = 10
INITIAL_FX = 1100

rows = []
for (mm_key, scenario_key, seed), (log_dir, status) in results.items():
    if status not in ('ok', 'cached'):
        continue
    mm_cfg = MMS[mm_key]
    a_pnl = get_pnl(log_dir, mm_cfg['a_usd_agent_name'])
    b_pnl = get_pnl(log_dir, mm_cfg['b_eur_agent_name'])
    if a_pnl is None or b_pnl is None:
        continue
    a_vol = total_mm_volume(log_dir, mm_cfg['a_usd_agent_name'])
    b_vol = total_mm_volume(log_dir, mm_cfg['b_eur_agent_name'])
    rebate_usd_cents = (a_vol + b_vol * INITIAL_FX / 1000) * POST_HOC_REBATE_CENTS

    total_starting = a_pnl['starting'] + b_pnl['starting']
    total_pnl = a_pnl['pnl'] + b_pnl['pnl'] + rebate_usd_cents
    rows.append({
        'mm': mm_key,
        'scenario': scenario_key,
        'seed': seed,
        'total_pnl_pct': total_pnl / total_starting * 100,
        'total_pnl_usd': total_pnl / 100,
        'A_pnl_pct': a_pnl['pnl_pct'],
        'B_pnl_pct': b_pnl['pnl_pct'],
        'mm_volume_shares': a_vol + b_vol,
        'rebate_usd': rebate_usd_cents / 100,
    })

df = pd.DataFrame(rows)

agg = df.groupby(['mm', 'scenario']).agg(
    pnl_mean=('total_pnl_pct', 'mean'),
    pnl_std=('total_pnl_pct', 'std'),
    pnl_min=('total_pnl_pct', 'min'),
    pnl_max=('total_pnl_pct', 'max'),
    n_seeds=('seed', 'count'),
    vol_mean=('mm_volume_shares', 'mean'),
    rebate_mean=('rebate_usd', 'mean'),
).reset_index()
print(f'Aggregate over seeds (rebate = {POST_HOC_REBATE_CENTS:.1f} c/share):')
display(agg.round(3))

Aggregate over seeds (rebate = 10.0 c/share):


,mm,scenario,pnl_mean,pnl_std,pnl_min,pnl_max,n_seeds,vol_mean,rebate_mean
0,adaptive,drift_down,-0.001,0.099,-0.184,0.191,25,19817.12,2080.398
1,adaptive,drift_up,0.002,0.098,-0.166,0.203,25,19345.12,2030.312
2,adaptive,fx_shock,0.023,0.105,-0.159,0.269,25,19517.48,2048.993
3,adaptive,liquidity_crisis,0.033,0.097,-0.132,0.225,25,19532.68,2050.482
4,adaptive,megashock,-0.013,0.111,-0.227,0.169,25,19961.84,2095.938
5,adaptive,news_shock,-0.011,0.114,-0.277,0.168,25,19639.64,2062.038
6,adaptive,stationary,0.017,0.109,-0.170,0.226,25,19615.88,2059.357
7,as,drift_down,0.092,0.118,-0.067,0.343,25,70091.80,7359.136
8,as,drift_up,0.027,0.105,-0.149,0.269,25,70303.40,7379.876
9,as,fx_shock,0.161,0.116,-0.065,0.418,25,69462.44,7293.222


## Результаты: финальные таблицы

1. **Mean PnL %** — главная метрика; цветная заливка (red→green) для быстрой
   визуальной оценки.
2. **Std PnL %** — разброс по seed'ам. Если std сравним с разностью между MM —
   статистической значимости недостаточно.

In [32]:
mm_order = list(MMS.keys())
sc_order = list(SCENARIOS.keys())

pivot_mean = agg.pivot(index='mm', columns='scenario', values='pnl_mean').reindex(mm_order)[sc_order]
print('=== Mean PnL % across seeds ===')
display(pivot_mean.style.format('{:>+7.3f}%').background_gradient(cmap='RdYlGn', axis=None, vmin=-0.5, vmax=0.5))

pivot_std = agg.pivot(index='mm', columns='scenario', values='pnl_std').reindex(mm_order)[sc_order]
print('\n=== Std PnL % across seeds ===')
display(pivot_std.style.format('{:>7.3f}%'))

pivot_vol = agg.pivot(index='mm', columns='scenario', values='vol_mean').reindex(mm_order)[sc_order]
print('\n=== Mean trade volume (shares per MM-pair) ===')
display(pivot_vol.style.format('{:>8,.0f}').background_gradient(cmap='Blues', axis=None))

print('\n=== Per-seed PnL %, full detail ===')
df_detail = df.pivot_table(
    index=['mm'], columns=['scenario', 'seed'], values='total_pnl_pct'
).reindex(mm_order)
display(df_detail.style.format('{:>+6.3f}'))

=== Mean PnL % across seeds ===


scenario,stationary,megashock,drift_down,drift_up,news_shock,liquidity_crisis,fx_shock
mm,,,,,,,
baseline,+0.138%,+0.081%,-0.201%,-0.425%,+0.082%,+0.139%,+0.134%
spread_based_tight,-0.016%,-0.070%,-0.315%,-0.214%,+0.011%,+0.020%,-0.011%
adaptive,+0.017%,-0.013%,-0.001%,+0.002%,-0.011%,+0.033%,+0.023%
as,+0.167%,+0.079%,+0.092%,+0.027%,+0.139%,+0.167%,+0.161%
glft,+0.227%,+0.119%,+0.207%,+0.210%,+0.193%,+0.236%,+0.243%
rl_glft,+0.230%,+0.196%,+0.228%,+0.225%,+0.210%,+0.226%,+0.234%



=== Std PnL % across seeds ===


scenario,stationary,megashock,drift_down,drift_up,news_shock,liquidity_crisis,fx_shock
mm,,,,,,,
baseline,0.144%,0.245%,0.359%,0.399%,0.236%,0.128%,0.161%
spread_based_tight,0.139%,0.163%,0.281%,0.242%,0.137%,0.125%,0.150%
adaptive,0.109%,0.111%,0.099%,0.098%,0.114%,0.097%,0.105%
as,0.126%,0.144%,0.118%,0.105%,0.098%,0.114%,0.116%
glft,0.113%,0.097%,0.106%,0.103%,0.091%,0.127%,0.118%
rl_glft,0.086%,0.103%,0.090%,0.091%,0.097%,0.094%,0.099%



=== Mean trade volume (shares per MM-pair) ===


scenario,stationary,megashock,drift_down,drift_up,news_shock,liquidity_crisis,fx_shock
mm,,,,,,,
baseline,"59,381","59,967","59,658","59,680","59,578","59,430","59,455"
spread_based_tight,"18,681","19,728","18,953","19,086","18,862","18,688","18,628"
adaptive,"19,616","19,962","19,817","19,345","19,640","19,533","19,517"
as,"69,780","69,707","70,092","70,303","69,690","69,523","69,462"
glft,"72,392","73,039","72,797","73,216","72,221","72,308","72,219"
rl_glft,"75,486","76,059","75,820","75,769","75,554","75,554","75,601"



=== Per-seed PnL %, full detail ===


## Ranking по сценариям

In [33]:
rank_rows = []
for sc in sc_order:
    col = pivot_mean[sc].dropna()
    if col.empty:
        continue
    best, worst = col.idxmax(), col.idxmin()
    best_std = agg[(agg['mm'] == best) & (agg['scenario'] == sc)]['pnl_std'].iloc[0]
    rank_rows.append({
        'scenario': sc,
        'best_mm': best, 'best_mean_pct': col[best], 'best_std': best_std,
        'worst_mm': worst, 'worst_mean_pct': col[worst],
        'spread': col[best] - col[worst],
    })
rank_df = pd.DataFrame(rank_rows).set_index('scenario')
display(rank_df.style.format({
    'best_mean_pct': '{:>+7.3f}%',
    'best_std':      '{:>7.3f}%',
    'worst_mean_pct':'{:>+7.3f}%',
    'spread':        '{:>7.3f}%',
}))

,best_mm,best_mean_pct,best_std,worst_mm,worst_mean_pct,spread
scenario,,,,,,
stationary,rl_glft,+0.230%,0.086%,spread_based_tight,-0.016%,0.245%
megashock,rl_glft,+0.196%,0.103%,spread_based_tight,-0.070%,0.266%
drift_down,rl_glft,+0.228%,0.090%,spread_based_tight,-0.315%,0.543%
drift_up,rl_glft,+0.225%,0.091%,baseline,-0.425%,0.650%
news_shock,rl_glft,+0.210%,0.097%,adaptive,-0.011%,0.221%
liquidity_crisis,glft,+0.236%,0.127%,spread_based_tight,+0.020%,0.217%
fx_shock,glft,+0.243%,0.118%,spread_based_tight,-0.011%,0.254%
